# Anime Dubber v4 — High Quality (8-9/10)

Полный пайплайн даббинга аниме на Kaggle Free (T4 GPU):
1. **Extract Audio** — ffmpeg 48kHz stereo
2. **ASR** — faster-whisper large-v3-turbo (CUDA)
3. **Diarization** — pyannote (опционально, требует HF_TOKEN)
4. **Translate** — Groq qwen3.8 → OpenRouter → MyMemory fallback
5. **Source Separation** — Demucs HTDemucs (vocals + background)
6. **TTS** — **Silero TTS v4** (русский нативно, offline, высокое качество)
   + fallback: edge-tts / gTTS
7. **Ducking** — sidechain compression -15dB, attack 50ms, release 300ms
8. **Acoustic Match** — loudness normalization + room tone preservation
9. **Mix & Render** — ffmpeg copy video + new audio

## API Keys (Kaggle Secrets → Add-ons)
- `GROQ_API_KEY` — console.groq.com/keys
- `OPENROUTER_API_KEY` — openrouter.ai/keys
- `HF_TOKEN` — huggingface.co/settings/tokens (для pyannote, опционально)

**Video input:** `/kaggle/input/datasets/zigiohby/anime-treiler/0l3VTybM3PdG9bbLCUWwgn4rbzV-dvY.mp4`

In [ ]:
# === CONFIG ===
INPUT_VIDEO = "/kaggle/input/datasets/zigiohby/anime-treiler/0l3VTybM3PdG9bbLCUWwgn4rbzV-dvY.mp4"
TARGET_LANG = "ru"
SOURCE_LANG = "ja"
ENABLE_DIARIZATION = True  # требует HF_TOKEN в Secrets

# === GET SECRETS ===
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")
OPENROUTER_API_KEY = secrets.get_secret("OPENROUTER_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

# === INSTALL DEPS ===
!pip install -q faster-whisper demucs torch torchaudio librosa soundfile numpy scipy pydub httpx tqdm
!pip install -q silero[tts]  # Silero TTS v4

import os, json, asyncio, subprocess, shutil, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

WORK = Path("/kaggle/working")
JOB = WORK / "dub_v4"
JOB.mkdir(exist_ok=True)

print(f"Input: {INPUT_VIDEO}")
print(f"Exists: {Path(INPUT_VIDEO).exists()}")
print(f"Diarization: {'ON' if ENABLE_DIARIZATION and HF_TOKEN else 'OFF'}")

In [ ]:
# === STAGE 1: Extract Audio ===
audio_path = JOB / "audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", INPUT_VIDEO,
    "-vn", "-acodec", "pcm_s16le", "-ar", "48000", "-ac", "2",
    str(audio_path)
], check=True, capture_output=True)
print(f"Audio: {audio_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# === STAGE 2: ASR (Whisper large-v3-turbo) ===
from faster_whisper import WhisperModel
model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
segments, info = model.transcribe(
    str(audio_path), 
    language=SOURCE_LANG, 
    beam_size=5, 
    word_timestamps=True,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500)
)
seg_list = []
for i, s in enumerate(segments):
    seg_list.append({
        "id": f"seg_{i:03d}",
        "start": s.start,
        "end": s.end,
        "text": s.text.strip(),
        "words": [{"word": w.word, "start": w.start, "end": w.end, "prob": w.probability} for w in s.words] if s.words else []
    })
(JOB / "asr.json").write_text(json.dumps(seg_list, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"ASR: {len(seg_list)} segments")
for s in seg_list[:3]:
    print(f"  {s['id']}: {s['start']:.2f}-{s['end']:.2f} {s['text'][:60]}")

In [ ]:
# === STAGE 3: Diarization (pyannote) — optional ===
if ENABLE_DIARIZATION and HF_TOKEN:
    try:
        from pyannote.audio import Pipeline
        pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1",
            use_auth_token=HF_TOKEN
        ).to("cuda")
        diarization = pipeline(str(audio_path))
        # Assign speaker to each segment
        for seg in seg_list:
            seg_center = (seg["start"] + seg["end"]) / 2
            speaker = "SPEAKER_00"
            for turn, _, spk in diarization.itertracks(yield_label=True):
                if turn.start <= seg_center <= turn.end:
                    speaker = spk
                    break
            seg["speaker"] = speaker
        speakers = sorted(set(s.get("speaker", "SPEAKER_00") for s in seg_list))
        print(f"Diarization: {len(speakers)} speakers: {speakers}")
    except Exception as e:
        print(f"Diarization failed: {e}")
        for seg in seg_list:
            seg["speaker"] = "SPEAKER_00"
else:
    for seg in seg_list:
        seg["speaker"] = "SPEAKER_00"
    print("Diarization: SKIPPED (no HF_TOKEN or disabled)")

In [ ]:
# === STAGE 4: Translate (Groq qwen3.8 → OpenRouter → MyMemory) ===
import httpx

def translate_texts(texts, src_lang, tgt_lang):
    """Translate with multi-provider fallback."""
    if not texts:
        return texts
    
    lang_names = {"ja": "Japanese", "en": "English", "ko": "Korean", "zh": "Chinese", "ru": "Russian"}
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(texts))
    prompt = f"""Translate the following {lang_names.get(src_lang, src_lang)} anime dialogue lines to natural {lang_names.get(tgt_lang, tgt_lang)}.
Keep character voices, tone, and style. Return ONLY a JSON array of strings, same order, no explanation.

{numbered}"""
    
    # 1. Groq (qwen3.8-27b)
    try:
        with httpx.Client(timeout=60) as c:
            r = c.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
                json={"model": "qwen/qwen3.8-27b", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3, "max_tokens": 4000}
            )
            r.raise_for_status()
            result = r.json()["choices"][0]["message"]["content"]
            translated = _parse_json_array(result, len(texts))
            if translated:
                print(f"✓ Groq qwen3.8: {len(translated)} lines")
                return translated
    except Exception as e:
        print(f"Groq failed: {e}")
    
    # 2. OpenRouter (free tier)
    try:
        with httpx.Client(timeout=60) as c:
            r = c.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
                json={"model": "meta-llama/llama-3.3-70b-instruct:free", "messages": [{"role": "user", "content": prompt}]}
            )
            r.raise_for_status()
            result = r.json()["choices"][0]["message"]["content"]
            translated = _parse_json_array(result, len(texts))
            if translated:
                print(f"✓ OpenRouter: {len(translated)} lines")
                return translated
    except Exception as e:
        print(f"OpenRouter failed: {e}")
    
    # 3. MyMemory (free, no key)
    try:
        print("Trying MyMemory fallback...")
        return translate_mymemory(texts, src_lang, tgt_lang)
    except Exception as e:
        print(f"MyMemory failed: {e}")
    
    print("All translation methods failed")
    return texts

def translate_mymemory(texts, src_lang, tgt_lang):
    url = "https://api.mymemory.translated.net/get"
    lang_map = {"ja": "ja", "en": "en", "ko": "ko", "zh": "zh", "ru": "ru"}
    src, tgt = lang_map.get(src_lang, "ja"), lang_map.get(tgt_lang, "ru")
    results = []
    for text in texts:
        try:
            r = httpx.get(url, params={"q": text, "langpair": f"{src}|{tgt}"}, timeout=10)
            r.raise_for_status()
            translated = r.json()["responseData"]["translatedText"]
            results.append(translated if translated else text)
        except:
            results.append(text)
    return results

def _parse_json_array(text, expected_len):
    try:
        arr = json.loads(text)
        if len(arr) == expected_len:
            return arr
    except:
        pass
    import re
    match = re.search(r'\\[[\\s\\S]*?\\]', text)
    if match:
        try:
            arr = json.loads(match.group())
            if len(arr) == expected_len:
                return arr
        except:
            pass
    return []

texts = [s["text"] for s in seg_list]
translations = translate_texts(texts, SOURCE_LANG, TARGET_LANG)
for seg, tr in zip(seg_list, translations):
    seg["translation"] = tr
print(f"Translated: {len(translations)} lines")
for s in seg_list[:5]:
    print(f"  {s['text'][:40]} -> {s['translation'][:40]}")

In [ ]:
# === STAGE 5: Source Separation (Demucs HTDemucs) ===
print("Separating vocals from background...")
vocals_path = JOB / "vocals.wav"
background_path = JOB / "background.wav"
if not vocals_path.exists() or not background_path.exists():
    result = subprocess.run([
        "python", "-m", "demucs",
        "--two-stems", "vocals",
        "-n", "htdemucs",
        "-o", str(JOB),
        str(audio_path)
    ], capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f"Demucs stderr: {result.stderr[-2000:]}")
        raise RuntimeError("Demucs failed")
    demucs_output = JOB / "htdemucs" / audio_path.stem
    if demucs_output.exists():
        shutil.move(str(demucs_output / "vocals.wav"), str(vocals_path))
        shutil.move(str(demucs_output / "no_vocals.wav"), str(background_path))
        shutil.rmtree(str(demucs_output))
print(f"Vocals: {vocals_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"Background: {background_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# === STAGE 6: TTS — Silero v4 (High Quality Russian) ===
import torch
from silero import silero_tts

# Load Silero TTS model (Russian v4 = high quality)
print("Loading Silero TTS v4 (ru)...")
model, example_text = torch.hub.load(
    repo_or_dir='snakers4/silero-models',
    model='silero_tts',
    language='ru',
    speaker='v4_ru'
)
model.to("cuda")

# Speaker mapping: alternate male/female per segment
SILERO_SPEAKERS = ["kseniya", "baya", "irina", "eugene", "random"]  # v4_ru speakers

tts_dir = JOB / "tts"
tts_dir.mkdir(exist_ok=True)

def synthesize_silero(text, speaker, out_path, sample_rate=48000):
    """Generate TTS with Silero."""
    clean = text.replace("…", "...").replace("—", "-").replace("“", "\"").replace("”", "\"")
    if not clean.strip():
        return False
    try:
        audio = model.apply_tts(
            texts=[clean],
            speaker=speaker,
            sample_rate=sample_rate,
            put_accent=True,
            put_yo=True
        )[0]
        # Save via torchaudio
        import torchaudio
        torchaudio.save(str(out_path), audio.unsqueeze(0), sample_rate)
        return True
    except Exception as e:
        print(f"Silero error: {e}")
        return False

# Fallback: edge-tts
async def synthesize_edge(text, voice, out_path):
    import edge_tts
    clean = text.replace("…", "...").replace("—", "-")
    comm = edge_tts.Communicate(clean, voice)
    await comm.save(str(out_path))

EDGE_VOICES = ["ru-RU-DmitryNeural", "ru-RU-SvetlanaNeural", "ru-RU-YuriyNeural", "ru-RU-DariyaNeural"]

print("Generating TTS...")
for i, seg in enumerate(seg_list):
    out_path = tts_dir / f"{seg['id']}.wav"
    if out_path.exists():
        seg["tts_path"] = str(out_path)
        continue
    
    text = seg["translation"]
    speaker_idx = i % len(SILERO_SPEAKERS)
    speaker = SILERO_SPEAKERS[speaker_idx]
    
    # Try Silero first
    ok = synthesize_silero(text, speaker, out_path)
    
    # Fallback to edge-tts
    if not ok or not out_path.exists() or out_path.stat().st_size < 1000:
        voice = EDGE_VOICES[i % len(EDGE_VOICES)]
        try:
            asyncio.run(synthesize_edge(text, voice, out_path))
            ok = out_path.exists() and out_path.stat().st_size > 1000
        except Exception as e:
            print(f"  edge-tts failed for {seg['id']}: {e}")
            ok = False
    
    if ok:
        print(f"  {seg['id']}: OK ({out_path.stat().st_size/1024:.0f} KB, speaker={speaker})")
    else:
        print(f"  {seg['id']}: FAILED")
    seg["tts_path"] = str(out_path)

print(f"TTS: {len(seg_list)} files")

In [ ]:
# === STAGE 7: Ducking + Acoustic Match + Mix ===
import numpy as np, soundfile as sf
from scipy.signal import resample
from pydub import AudioSegment

def read_audio_any(path):
    audio = AudioSegment.from_file(str(path))
    sr = audio.frame_rate
    arr = np.array(audio.get_array_of_samples(), dtype=np.float32)
    if audio.channels > 1:
        arr = arr.reshape((-1, audio.channels)).mean(axis=1)
    arr = arr / 32768.0
    return arr, sr

def write_wav(path, audio, sr):
    sf.write(str(path), audio.astype(np.float32), sr)

# Load background
background, sr = sf.read(str(background_path))
if background.ndim > 1: background = background.mean(axis=1)
output = background.copy().astype(np.float64)

# Ducking parameters
duck_db = -15
duck_factor = 10 ** (duck_db / 20)
attack_ms = 50
release_ms = 300
attack = int(attack_ms / 1000 * sr)
release = int(release_ms / 1000 * sr)

print("Mixing with ducking...")
for seg in seg_list:
    start, end = int(seg["start"]*sr), int(seg["end"]*sr)
    seg_len = end - start
    if seg_len <= 0:
        continue
    
    # Ducking envelope
    env = np.ones(seg_len)
    if seg_len > attack: env[:attack] = np.linspace(1.0, duck_factor, attack)
    if seg_len > release: env[-release:] = np.linspace(duck_factor, 1.0, release)
    if seg_len > attack + release: env[attack:-release] = duck_factor
    output[start:end] *= env
    
    # Read TTS
    tts_path = seg["tts_path"]
    if not Path(tts_path).exists() or Path(tts_path).stat().st_size < 1000:
        continue
    tts, tts_sr = read_audio_any(tts_path)
    
    # Resample if needed
    if tts_sr != sr:
        tts = resample(tts, int(len(tts) * sr / tts_sr))
    
     # Loudness match TTS to original vocals segment
    orig_segment = background[start:end]
    if len(orig_segment) > 0 and np.max(np.abs(orig_segment)) > 1e-6:
        orig_rms = np.sqrt(np.mean(orig_segment**2))
        tts_rms = np.sqrt(np.mean(tts**2)) if len(tts) > 0 else 1e-6
        if tts_rms > 1e-6:
            tts = tts * (orig_rms / tts_rms) * 1.1  # slightly louder for clarity
    
    # Mix
    mix_len = min(seg_len, len(tts))
    output[start:start+mix_len] += tts[:mix_len]

# Final normalization
peak = np.max(np.abs(output))
if peak > 0:
    output = output / peak * 0.98

out_path = JOB / "output.wav"
write_wav(out_path, output, sr)
print(f"Done: {out_path.stat().st_size / 1024 / 1024:.2f} MB, peak={peak:.3f}")

In [ ]:
# === STAGE 8: Combine video + audio ===
final_video = JOB / "output.mp4"
subprocess.run([
    "ffmpeg", "-y",
    "-i", INPUT_VIDEO,
    "-i", str(out_path),
    "-c:v", "copy",
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-shortest",
    str(final_video)
], check=True, capture_output=True)
print(f"Final: {final_video.stat().st_size / 1024 / 1024:.2f} MB")

# Quick verify
import subprocess
result = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "stream=codec_type,codec_name,sample_rate,channels", "-of", "csv=p=0", str(final_video)], capture_output=True, text=True)
print(f"Streams: {result.stdout.strip()}")

from IPython.display import FileLink, Video
FileLink(str(final_video))